# 4) Run GPEs #

Code for running the ECO-FAST and ORG-FAST analysis using gaussian process regression. <br>
<br>
Inputs: <br>
Metrics_PFT.nc - created by notebook 3 and contains metrics for GPP, Stress, and Drought Sensitivity for the primary PFT of each grid cell (y vars in the ORG-FAST analysis) <br>
Metrics_GC.nc - created by notebook 3 and contains metrics for Drought Sensitivity for entire grid cell (y vars in the ECO-FAST analysis)<br>
Traits.nc - created by notebook 2 and contains the parameter values, both for individual PFTs and gridcell weighted means and coefficient of variations (y vars in both analyses)<br>
primaryPFT.nc - a helper dataset for selecting the most abundant pft (based on cveg) in each grid cell <br>
<br>
Ouputs:<br>
Metrics.nc - contains select output variables used for analysis

## Load packages ##

In [1]:
import importlib.util
import os
import pickle
import warnings
from math import pi
import cartopy.crs as ccrs
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap
from pypalettes import load_cmap
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from tqdm import tqdm
import gpflow
from esem import gp_model
from esem.sampler import MCMCSampler
from esem.utils import get_random_params, leave_one_out, prediction_within_ci
from SALib.analyze import fast
from SALib.sample import fast_sampler
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

repo_dir = os.getcwd()
repo_dir = os.path.dirname(repo_dir)

2025-09-02 12:44:02.859677: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-02 12:44:02.952535: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-02 12:44:02.952567: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-02 12:44:02.953488: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-02 12:44:03.028542: I tensorflow/core/platform/cpu_feature_guar

## Functions ##

In [2]:
def gpe(data, xvar, yvar):
    # two scalers: one for X, one for y
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    # Data prep: aggregate over 'year' if present, then convert to DataFrame
    if 'year' in data.dims:
        subset = data.mean(dim='year', skipna=True)
    else:
        subset = data
    mean_y = data[yvar].mean(skipna=True).values.item()
    std_y = data[yvar].std(skipna=True).values.item()
    data[yvar] = data[yvar].where(np.abs(data[yvar] - mean_y) <= 3 * std_y)
    
    df = subset[xvar + [yvar]].to_dataframe().reset_index()
    df = df.replace([np.inf, -np.inf], np.nan)#.dropna()

    # preserve ensemble ID, drop gridcell
    ens = df['ens']
    df = df.drop(['gridcell','ens'], axis=1)

    # — Z-score X and y —
    X_all = pd.DataFrame(
        scaler_x.fit_transform(df[xvar]),
        columns=xvar
    )
    if yvar == 'Stress':
        y_all = df[[yvar]]
    else:
        y_all = pd.DataFrame(
            scaler_y.fit_transform(df[[yvar]]),
            columns=[yvar]
        )
    
    df_scaled = pd.concat([X_all, y_all], axis=1)
    df_scaled['ens'] = ens.values

    # split train and test by ensemble ID
    ids = list(range(1,501))
    train_ids, test_ids = train_test_split(ids, test_size=0.3, random_state=39)
    df_tr = df_scaled[df_scaled.ens.isin(train_ids)].dropna()
    df_te = df_scaled[df_scaled.ens.isin(test_ids)].dropna()

    X_train = df_tr[xvar].values
    y_train = df_tr[yvar].values.reshape(-1,1)
    X_test  = df_te[xvar].values
    y_test  = df_te[yvar].values

    # — Build & train the GP model —
    D = len(xvar)
    kernel = (
        gpflow.kernels.Linear(active_dims=range(D), variance=1.0)
        + gpflow.kernels.Matern32(active_dims=range(D),
                                 variance=1.0,
                                 lengthscales=np.ones(D))
    )
    model = gpflow.models.GPR((X_train, y_train), kernel=kernel)
    opt = gpflow.optimizers.Scipy()
    opt.minimize(model.training_loss,
                 model.trainable_variables,
                 options=dict(maxiter=500))

    # — Evaluate R²  —
    μ_tr, _ = model.predict_y(X_train)
    μ_te, _ = model.predict_y(X_test)
    r2_train = r2_score(y_train.flatten(), μ_tr.numpy().flatten())
    r2_test  = r2_score(y_test,      μ_te.numpy().flatten())

    # — FAST sensitivity  —
    xdata = pd.DataFrame(df_scaled.drop(columns=[yvar,'ens']), columns=xvar)
    bounds = [[xdata[c].quantile(0.1), xdata[c].quantile(0.9)+1e-6] for c in xvar]
    problem = {'names': xvar, 'num_vars': D, 'bounds': bounds}
    sample = fast_sampler.sample(problem, 1000, M=4)
    Y, _ = model.predict_f(sample)
    FASTres = fast.analyze(problem, Y.numpy().flatten(), M=4,
                           num_resamples=100, conf_level=0.95,
                           print_to_console=False)
    Si_df = pd.DataFrame.from_dict(FASTres).set_index('names')\
              .sort_values('S1', ascending=False)

    # — Direct GP slopes via analytic gradients —
    X_tf = tf.convert_to_tensor(df_scaled[xvar].values, dtype=tf.float64)
    with tf.GradientTape() as tape:
        tape.watch(X_tf)
        μ, _ = model.predict_f(X_tf)       # [N,1]
    grads = tape.batch_jacobian(μ, X_tf)   # [N,1,D]
    grads = tf.squeeze(grads, axis=1)      # [N,D]

    grads_xr = xr.DataArray(grads.numpy(), dims=['ens', 'trait'],
                        coords={'trait': xvar})

    X_tf_xr = xr.DataArray(X_tf.numpy(), dims=['ens', 'trait'],
                       coords={'trait': xvar})

    
    avg_slopes = tf.reduce_mean(grads, axis=0).numpy()

    slope_series = pd.Series(avg_slopes, index=xvar,
                             name='GP_avg_slope')\
                     .sort_values(ascending=False)

    return r2_test, r2_train, Si_df, slope_series, X_tf_xr, grads_xr

def gpe_gridcell(i, data, xvar, yvar):
    try:
        if i%25 == 0:
            print(i)
        xy = data.sel(gridcell=i)
        r2_test_gp, r2_train_gp, df_Si, df_slope, xr_Xtest, xr_PSlope = gpe(xy, xvar, yvar)
        return {
            'gridcell': i,
            'r2_test': r2_test_gp,
            'r2_train': r2_train_gp,
            'importances': df_Si,
            'PDP_Slope': df_slope,
            'XTest': xr_Xtest,
            'XSlope': xr_PSlope
        }
    except Exception as e:
        return {
            'gridcell': i,
            'r2_test': np.nan,
            'r2_train': np.nan,
            'importances': np.nan,
            'PDP_Slope': np.nan,
            'XTest': np.nan,
            'XSlope': np.nan
        }

def gpe_pft(i, data, xvars, yvar):
    try:
        if i%25 == 0:
            print(i)
        ds = data.sel(gridcell = i)
        prim_pft_gc = prim_pft.sel(gridcell = i).pft.values
        ds = ds.sel(pft = prim_pft_gc)

        if 'year' in ds[yvar].dims:
            ds = ds.mean(dim = 'year')

        r2_test_gp, r2_train_gp, df_Si, df_slope, xr_Xtest, xr_PSlope = gpe(ds, xvars, yvar)
        
        return {
            'gridcell': i,
            'r2_test': r2_test_gp,
            'r2_train': r2_train_gp,
            'importances': df_Si,
            'PDP_Slope': df_slope,
            'XTest': xr_Xtest,
            'XSlope': xr_PSlope
        }
    except Exception as e:
        return {
            'gridcell': i,
            'r2_test': np.nan,
            'r2_train': np.nan,
            'importances': np.nan,
            'PDP_Slope': np.nan,
            'XTest': np.nan,
            'XSlope': np.nan
        }
        
def gppp_slope(r2_train_list, r2_test_list, slope_list, importances_list_gp, X_tf_xr_list, grads_xr_list, xvar, gridcells):
    # Create a DataFrame filled with NaNs for handling missing entries
    nan_df = pd.DataFrame(np.nan, index=xvar, columns=['S1', 'ST', 'S1_conf', 'ST_conf'])
    
    # Convert each DataFrame to xarray and handle NaNs
    data_arrays = []
    for df in importances_list_gp:
        if isinstance(df, pd.DataFrame):
            data_array = df.to_xarray().rename({'names':'trait'})
        else:
            data_array = nan_df.to_xarray().rename({'index':'trait'})
        data_arrays.append(data_array)
    
    # Concatenate along a new dimension 'gridcell'
    combined = xr.concat(data_arrays, dim='gridcell')

    nan_df = pd.DataFrame(np.nan, index=xvar, columns=['slope'])
    slope_arrays = []
    for s in slope_list:
        if isinstance(s, pd.Series):
            # ensure name is 'slope' so to_xarray() yields a DataArray called 'slope'
            s2 = s.copy()
            s2.name = 'slope'
            da = s2.to_xarray().rename({'index': 'trait'}).rename('slope')
        else:
            da = nan_df.to_xarray().rename({'index': 'trait'}).slope
        slope_arrays.append(da)

    slope_combined = xr.concat(slope_arrays, dim='gridcell')
    combined['slope'] = slope_combined

    # Add X_tf_xr (input values) and grads_xr (GP slopes)
    nan_X = xr.DataArray(np.full((500, len(xvar)), np.nan), dims=['ens', 'trait'], coords={'trait': xvar})
    X_arrays, G_arrays = [], []
    for x, g in zip(X_tf_xr_list, grads_xr_list):
        X_arrays.append(x if isinstance(x, xr.DataArray) else nan_X)
        G_arrays.append(g if isinstance(g, xr.DataArray) else nan_X)
    X_combined = xr.concat(X_arrays, dim='gridcell')
    G_combined = xr.concat(G_arrays, dim='gridcell')
    combined['X'] = X_combined
    combined['grads'] = G_combined
    combined['r2_train'] = xr.DataArray(r2_train_list, dims=['gridcell'], coords={'gridcell': gridcells})
    combined['r2_test'] = xr.DataArray(r2_test_list, dims=['gridcell'], coords={'gridcell':gridcells})

    
    return combined

## Input Data ##

In [3]:
# trait data
trait_data = xr.open_dataset(repo_dir+'/input/Traits.nc')
trait_data

<xarray.Dataset>
Dimensions:                   (gridcell: 400, pft: 15, ens: 500)
Coordinates:
  * gridcell                  (gridcell) int64 0 1 2 3 4 ... 395 396 397 398 399
  * pft                       (pft) object 'not_vegetated' ... 'c4_grass'
  * ens                       (ens) int64 1 2 3 4 5 6 ... 496 497 498 499 500
Data variables: (12/78)
    froot_leaf                (ens, pft) float64 ...
    kmax                      (ens, pft) float64 ...
    krmax                     (ens, pft) float64 ...
    leaf_long                 (ens, pft) float64 ...
    leafcn                    (ens, pft) float64 ...
    lmr_intercept_atkin       (ens, pft) float64 ...
    ...                        ...
    stem_leafMean             (gridcell, ens) float64 ...
    stem_leafSD               (gridcell, ens) float64 ...
    stem_leafCV               (gridcell, ens) float64 ...
    theta_cjMean              (gridcell, ens) float64 ...
    theta_cjSD                (gridcell, ens) float64 ...
    theta_cjCV                (gridcell, ens) float64 ...

### PFT Level ###

In [4]:
xtraits_PFT = ['kmax_Norm','leafcn_Norm','medlynslope_Norm','psi50_Norm','slatop_Norm','jmaxb0','jmaxb1','wc2wjb0']
prim_pft=xr.open_dataset(repo_dir+'/utils/primaryPFT.nc')

Metrics_PFT = xr.open_dataset(repo_dir+'/supplement/input/DroughtSensVersions_Supp_PFT.nc')

inds_test_PFT = xr.merge([trait_data[xtraits_PFT], Metrics_PFT])
rename_dict = {var: var.replace("_Norm", "") for var in inds_test_PFT.data_vars}
inds_test_PFT = inds_test_PFT.rename(rename_dict)
xtraits_PFT = ['kmax','leafcn','medlynslope','psi50','slatop','jmaxb0','jmaxb1','wc2wjb0']
inds_test_PFT

<xarray.Dataset>
Dimensions:      (pft: 15, ens: 500, gridcell: 400)
Coordinates:
  * pft          (pft) object 'broadleaf_deciduous_boreal_shrub' ... 'not_veg...
  * ens          (ens) int64 1 2 3 4 5 6 7 8 ... 493 494 495 496 497 498 499 500
  * gridcell     (gridcell) int64 0 1 2 3 4 5 6 ... 393 394 395 396 397 398 399
Data variables:
    kmax         (ens, pft) float64 ...
    leafcn       (ens, pft) float64 ...
    medlynslope  (ens, pft) float64 ...
    psi50        (ens, pft) float64 ...
    slatop       (ens, pft) float64 ...
    jmaxb0       (ens) float64 ...
    jmaxb1       (ens) float64 ...
    wc2wjb0      (ens) float64 ...
    GPPMinYears  (gridcell, pft, ens) float64 ...
    TMinYears    (gridcell, pft, ens) float64 ...
    GPPSlope     (pft, gridcell, ens) float64 ...
    TSlope       (pft, gridcell, ens) float64 ...

### Grid Cell Level ###

In [5]:
xtraits_GC = ['kmaxCV', 'leafcnCV', 'medlynslopeCV', 'psi50CV', 'slatopCV', 'kmaxMean', 'leafcnMean', 'medlynslopeMean', 'psi50Mean', 'slatopMean', 'jmaxb0','jmaxb1','wc2wjb0']
Metrics_GC = xr.open_dataset(repo_dir+'/supplement/input/DroughtSensVersions_Supp_GC.nc')

inds_test_GC = xr.merge([trait_data[xtraits_GC], Metrics_GC])
inds_test_GC

<xarray.Dataset>
Dimensions:          (gridcell: 400, ens: 500)
Coordinates:
  * gridcell         (gridcell) int64 0 1 2 3 4 5 6 ... 394 395 396 397 398 399
  * ens              (ens) int64 1 2 3 4 5 6 7 8 ... 494 495 496 497 498 499 500
Data variables: (12/17)
    kmaxCV           (gridcell, ens) float64 ...
    leafcnCV         (gridcell, ens) float64 ...
    medlynslopeCV    (gridcell, ens) float64 ...
    psi50CV          (gridcell, ens) float64 ...
    slatopCV         (gridcell, ens) float64 ...
    kmaxMean         (gridcell, ens) float64 ...
    ...               ...
    jmaxb1           (ens) float64 ...
    wc2wjb0          (ens) float64 ...
    GPPMinYears      (gridcell, ens) float64 ...
    TMinYears        (gridcell, ens) float64 ...
    GPPSlope         (gridcell, ens) float64 ...
    TSlope           (gridcell, ens) float64 ...

## Run ORG-FAST Analysis ##

In [6]:
#GPP 3 Year
results_list = []
for i in range(0,400):
    result = gpe_pft(i, inds_test_PFT, xtraits_PFT, 'GPPMinYears')
    results_list.append(result)

# Run some post processing
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_PFT, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ORGFAST_GPPDroughtSens3Year.nc')

0


2025-08-22 12:42:48.280443: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


2025-08-22 12:43:14.945347: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:43:49.019248: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:44:09.712092: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:44:11.318041: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:44:29.863596: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

25


2025-08-22 12:44:43.701281: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:44:59.226246: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:45:09.681141: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 12:45:32.358379: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


...

350


2025-08-22 13:04:42.757637: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-08-22 13:04:47.545061: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_

375


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

In [6]:
#T 3 Year
results_list = []
for i in range(0,400):
    result = gpe_pft(i, inds_test_PFT, xtraits_PFT, 'TMinYears')
    results_list.append(result)

# Run some post processing
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_PFT, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ORGFAST_TDroughtSens3Year.nc')

0


2025-09-02 12:44:35.061134: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


2025-09-02 12:44:48.982161: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:44:53.252956: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:44:53.262311: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:44:53.271537: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:45:17.325085: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

25


2025-09-02 12:45:34.062884: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:45:40.663001: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:45:41.592614: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:45:41.641739: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 12:45:58.143694: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

...

/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

375


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/SALib/analyze/fast.py:117: RuntimeWarning: invalid value encountered in scalar divide
  return (D1 / V), (1.0 - Dt / V)
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/gl

/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

2025-09-02 12:59:06.443440: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

In [7]:
#GPP Slope
results_list = []
for i in range(0,400):
    result = gpe_pft(i, inds_test_PFT, xtraits_PFT, 'GPPSlope')
    results_list.append(result)

# Run some post processing
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_PFT, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ORGFAST_GPPDroughtSensSlope.nc')

0
25


2025-09-02 13:36:36.732337: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:37:11.671280: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:37:12.240044: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:37:12.976817: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:37:13.000874: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

50


2025-09-02 13:37:21.188356: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:37:26.057310: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_

75


...

2025-09-02 13:45:22.460863: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:45:22.468583: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_

350


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

375


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

In [8]:
#T Slope
results_list = []
for i in range(0,400):
    result = gpe_pft(i, inds_test_PFT, xtraits_PFT, 'TSlope')
    results_list.append(result)

# Run some post processing
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_PFT, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ORGFAST_TDroughtSensSlope.nc')

0


2025-09-02 13:47:09.536183: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:10.377410: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:10.428519: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


25


2025-09-02 13:47:15.906969: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:16.976624: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:17.981019: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:35.470799: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:47:44.909599: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

50


...

2025-09-02 13:55:57.937747: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:55:58.740303: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:55:58.776316: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:56:05.639712: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:56:20.530589: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

350


2025-09-02 13:56:43.695047: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

375


2025-09-02 13:57:03.047461: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

## Run ECO-FAST Analysis ##

In [ ]:
results_list = []
for i in range(0,400):
    result = gpe_gridcell(i, inds_test_GC, xtraits_GC, 'GPPMinYears')
    results_list.append(result)

# Convert results into lists
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_GC, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ECOFAST_GPPDroughtSens3Year.nc')

In [9]:
results_list = []
for i in range(0,400):
    result = gpe_gridcell(i, inds_test_GC, xtraits_GC, 'TMinYears')
    results_list.append(result)

# Convert results into lists
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_GC, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ECOFAST_TDroughtSens3Year.nc')

0


2025-09-02 13:57:25.347439: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:57:26.737597: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:57:27.472153: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:57:27.497037: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:57:29.581302: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

25


2025-09-02 13:58:18.066750: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:58:18.615253: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:58:19.977399: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:58:27.790739: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 13:58:30.494181: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

50


...

2025-09-02 14:12:26.751762: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:12:45.076331: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:12:45.086453: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:12:45.934907: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:12:45.962441: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular out

350


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / 

375


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / 

In [10]:
results_list = []
for i in range(0,400):
    result = gpe_gridcell(i, inds_test_GC, xtraits_GC, 'GPPSlope')
    results_list.append(result)

# Convert results into lists
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_GC, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ECOFAST_GPPDroughtSensSlope.nc')

0


2025-09-02 14:15:25.085409: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:15:27.633539: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:15:31.342888: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:15:47.454955: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


25


2025-09-02 14:15:56.811464: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:16:05.734044: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


50
75


...

2025-09-02 14:30:08.235461: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

350


2025-09-02 14:30:40.320986: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

375


/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype

In [11]:
results_list = []
for i in range(0,400):
    result = gpe_gridcell(i, inds_test_GC, xtraits_GC, 'TSlope')
    results_list.append(result)

# Convert results into lists
r2_train_list_gp = [r['r2_train'] for r in results_list]
r2_test_list_gp  = [r['r2_test'] for r in results_list]
importances_list_gp = [r['importances'] for r in results_list]
slope_list_gp = [r['PDP_Slope'] for r in results_list]
XTest_list_gp = [r['XTest'] for r in results_list]
XSlope_list_gp = [r['XSlope'] for r in results_list]

gp_output = gppp_slope(r2_train_list_gp, r2_test_list_gp, slope_list_gp, importances_list_gp, XTest_list_gp, XSlope_list_gp, xtraits_GC, range(0,400))
gp_output.to_netcdf(repo_dir+'/supplement/output/ECOFAST_TDroughtSensSlope.nc')

0


2025-09-02 14:31:48.621829: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:31:51.357161: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


25


2025-09-02 14:32:31.286512: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.


50
75


...

2025-09-02 14:44:54.625226: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne

350


2025-09-02 14:45:44.926042: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:45:45.757901: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
2025-09-02 14:45:45.788533: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_

375


2025-09-02 14:46:10.838639: W tensorflow/core/kernels/linalg/cholesky_op.cc:56] Cholesky decomposition was not successful. Eigen::LLT failed with error code 1. Filling lower-triangular output with NaNs.
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/glade/u/home/emargiotta/AridityEnv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / ne